# Week 34, Day 2: Trial-to-trial variability: Fano factors and variability quenching

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w034/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Variability is quenched

Spikes from 300 trials are simulated from -500 to 1000 ms with the stimulus at 0: 10 Hz before and 40 Hz after, each trial's rate multiplied by a gamma-distributed gain whose CV is 0.5 before and 0.15 after. Count spikes in 200 ms windows stepped by 50 ms, and compute the PSTH (Hz) and Fano factor in each window.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2010)
trials, dt = 300, 1.0
t = np.arange(-500, 1000, dt)

def gains(cv, n):
    k = 1 / cv**2
    return r.gamma(k, 1 / k, n)                      # mean 1, given CV

g_pre, g_post = gains(0.5, trials), gains(0.15, trials)
rate = np.where(t < 0, 10.0, 40.0)                   # Hz
lam = np.where(t[None, :] < 0, g_pre[:, None], g_post[:, None]) * rate[None, :] / 1000 * dt
spikes = r.uniform(size=lam.shape) < lam             # one Bernoulli draw per ms

starts = np.arange(-500, 801, 50)
centres = starts + 100
counts = np.array([spikes[:, (t >= s) & (t < s + 200)].sum(1) for s in starts]).T   # trials x windows
psth = ...
fano = ...
for c, p, f in zip(centres[::4], psth[::4], fano[::4]):
    print(f"window centred at {c:5.0f} ms: {p:5.1f} Hz, Fano {f:.2f}")
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(centres, psth); ax[0].set_ylabel("Hz"); ax[1].plot(centres, fano); ax[1].axhline(1, color="gray", lw=0.5)
ax[1].set_ylabel("Fano"); ax[1].set_xlabel("ms from stimulus"); plt.show()

In [ ]:
import numpy as np
def test_psth():
    "psth is the mean count per window divided by the window length, in Hz"
    assert np.allclose(psth, counts.mean(0) / 0.2)

def test_quench():
    "the Fano factor is above 1.3 before the stimulus and below 1.25 after it"
    assert fano[centres < -100].mean() > 1.3 and fano[centres > 300].mean() < 1.25

test_psth()
test_quench()
print("All checks passed")

### Exercise: How variable is the rate?

Using counts in a window before the stimulus (starting at -400 ms) and one after (starting at 400 ms), estimate the variance of the underlying rate as Var(N) minus mean(N), and convert it to a coefficient of variation of the rate, sqrt(variance)/mean, stored in `rate_cv['pre']` and `rate_cv['post']`.

In [ ]:
rate_cv = {}
for name, s in (("pre", -400), ("post", 400)):
    n = counts[:, list(starts).index(s)]
    var_rate = ...
    rate_cv[name] = ...
    print(f"{name:4s}: mean count {n.mean():.2f}, variance {n.var():.2f}, rate variance {var_rate:.2f}, rate CV {rate_cv[name]:.2f}")

In [ ]:
import numpy as np
def test_estimate():
    "the estimated rate CV is within 0.1 of the true gain CV before (0.5) and after (0.15) onset"
    assert abs(rate_cv["pre"] - 0.5) < 0.1 and abs(rate_cv["post"] - 0.15) < 0.1

test_estimate()
print("All checks passed")

### Exercise: Fano rises with the mean

For mean counts of 1, 4, 8 and 16 with a fixed rate CV of 0.3, simulate 4,000 trials of a doubly stochastic neuron (gamma gain, Poisson counts) and compute its Fano factor, `F_sim`. Compare with 1 + c² μ.

In [ ]:
means = [1, 4, 8, 16]
F_sim = []
for m in means:
    lam_trial = m * gains(0.3, 4000)
    n = ...
    F_sim.append(n.var() / n.mean())
    print(f"mean {m:2d}: Fano {F_sim[-1]:.2f}, theory {1 + 0.09 * m:.2f}")

In [ ]:
import numpy as np
def test_rise():
    "with a fixed rate CV of 0.3, the Fano factor rises with the mean count and matches 1 + c^2 mu within 0.1"
    assert all(a < b for a, b in zip(F_sim, F_sim[1:]))
    assert all(abs(f - (1 + 0.09 * m)) < 0.1 for f, m in zip(F_sim, means))

test_rise()
print("All checks passed")